In [ ]:
import os
import glob
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

# Define paths (Using capital 'Data' as seen in your folder structure)
class_0_dir = os.path.join("Data", "Class_0")
class_1_dir = os.path.join("Data", "Class_1")

# Load image paths
non_oil_images = glob.glob(os.path.join(class_0_dir, "*.*"))
oil_images = glob.glob(os.path.join(class_1_dir, "*.*"))

# --- 1. CLASS DISTRIBUTION ANALYSIS ---
print("--- 1. Class Distribution Analysis ---")
non_oil_count = len(non_oil_images)
oil_count = len(oil_images)

labels = ['Non-Oil Spill (Class 0)', 'Oil Spill (Class 1)']
sizes = [non_oil_count, oil_count]
colors = ['#66b3ff', '#ff9999']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Class Distribution (Percentage)')
axes[1].bar(labels, sizes, color=colors)
axes[1].set_title('Class Distribution (Count)')
axes[1].set_ylabel('Number of Images')
plt.tight_layout()
plt.show()

# --- 2. IMAGE VISUALIZATION ---
print("--- 2. Image Visualization ---")
def plot_random_grid(image_paths, title):
    plt.figure(figsize=(12, 6))
    plt.suptitle(title, fontsize=16)
    selected = random.sample(image_paths, 10)
    for i, img_path in enumerate(selected):
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        ax = plt.subplot(2, 5, i + 1)
        plt.imshow(img, cmap='gray')
        plt.axis("off")
    plt.tight_layout()
    plt.show()

plot_random_grid(non_oil_images, "Non-Oil Spill (Class 0) - Rough, bright surface")
plot_random_grid(oil_images, "Oil Spill (Class 1) - Dark patches (capillary wave dampening)")

# --- 3. IMAGE STATISTICS AND OUTLIER DETECTION ---
print("--- 3. Image Statistics and Outlier Detection ---")
def get_image_statistics(image_paths, class_label):
    data = []
    for img_path in image_paths:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            data.append({
                'filepath': img_path,
                'class': class_label,
                'mean_intensity': np.mean(img),
                'std_intensity': np.std(img)
            })
    return data

print("Calculating pixel statistics... (This may take a minute)")
stats_0 = get_image_statistics(non_oil_images, 'Non-Oil (Class 0)')
stats_1 = get_image_statistics(oil_images, 'Oil Spill (Class 1)')

df = pd.DataFrame(stats_0 + stats_1)

# Plot Boxplots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
df.boxplot(column='mean_intensity', by='class', ax=axes[0], grid=False)
axes[0].set_title('Distribution of Mean Pixel Intensity')
axes[0].set_ylabel('Mean Intensity')
axes[0].set_xlabel('')

df.boxplot(column='std_intensity', by='class', ax=axes[1], grid=False)
axes[1].set_title('Distribution of Standard Deviation')
axes[1].set_ylabel('Standard Deviation')
axes[1].set_xlabel('')
plt.suptitle('Image Statistics Boxplots')
plt.tight_layout()
plt.show()

# Display Outliers
lowest_mean_row = df.loc[df['mean_intensity'].idxmin()]
highest_mean_row = df.loc[df['mean_intensity'].idxmax()]

def display_outlier(row, title):
    img = cv2.imread(row['filepath'], cv2.IMREAD_GRAYSCALE)
    plt.imshow(img, cmap='gray')
    plt.title(f"{title}\nClass: {row['class']}\nMean: {row['mean_intensity']:.2f}")
    plt.axis('off')

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
display_outlier(lowest_mean_row, "Extreme Low Outlier (Darkest)")
plt.subplot(1, 2, 2)
display_outlier(highest_mean_row, "Extreme High Outlier (Brightest)")
plt.tight_layout()
plt.show()

### Findings and Interpretation

*   **1. Class Distribution (Dataset Balance):** The data is imbalanced. About 67% of the images are normal sea, and only 33% are oil spills. 
    * *Why it matters:* An AI could become biased toward guessing "normal sea," so we have to balance the data during training.

*   **2. Visual Patterns:** Oil slicks smooth out the ocean's surface. 
    * *Why it matters:* Smooth water reflects less radar, making oil spills look like **distinct dark patches**. Normal rough water reflects more radar, looking bright gray.

*   **3. Pixel Statistics:** The math matches our eyes. Oil spill images have a significantly lower mean pixel intensity (they are darker on average). Standard deviation helps show the sharp contrast between the dark oil and bright water.

*   **4. Outliers (Unusual Images):**
    * *Super Dark Outliers:* Usually caused by perfectly calm, windless water. This creates a "look-alike" false alarm that looks like oil but isn't.
    * *Super Bright Outliers:* Usually caused by metal ships, oil platforms, or severe storms bouncing a massive amount of radar back to the satellite.

**Conclusion:** 
We can successfully detect oil spills by looking for these dark patches and low average pixel intensities. However, to build a reliable AI, we must handle the dataset imbalance and teach the model to ignore false alarms like windless water.